# OrientaBot · Avance de proyecto 1
**Materia:** Aplicaciones con Inteligencia Artificial · 
**Estudiante:** Orlando Salazar

Asistente escolar **basado en reglas** con Python y NLTK, con **árboles de decisión** (memoria de conversación)
y pruebas con **pseudoestudiantes**. Información demostrativa: no contiene fechas ni requisitos oficiales.

**Cómo usarlo:** Entorno de ejecución → **Ejecutar todas**. Cada sección imprime su evidencia.

## 1. Definición de la solución
- **Problema:** los estudiantes repiten las mismas preguntas en Servicios Escolares (inscripción, becas,
  constancias, biblioteca…), lo que satura al personal y los hace esperar.
- **Objetivo general:** un chatbot de reglas que oriente sobre esas consultas frecuentes las 24 horas.
- **Usuarios:** estudiantes y aspirantes.
- **Alcance (Fase 1):** 12 intenciones, solo texto. Orienta y remite al área; no hace trámites ni consulta expedientes.
  Fases siguientes: clasificador de intenciones con ML y validación de documentos por imagen.
- **Beneficios esperados:** respuesta inmediata, menos carga para el personal, información consistente.

## 2. Arquitectura conceptual
Usuario → texto → normalización → **motor de reglas NLTK** (intención) → ¿tema con árbol?
→ sí: **gestor de diálogo** (nodo actual + mensaje pendiente) → pregunta o respuesta final
→ no: respuesta directa de la **base de conocimiento** → usuario. La regla `desconocida` es el fallback.

In [ ]:
import sys
assert sys.version_info >= (3, 10), 'Se requiere Python 3.10 o superior.'
%pip -q install nltk==3.9.1
import nltk                                  # se importa después de instalarlo
print('Python:', sys.version.split()[0])
print('NLTK:  ', nltk.__version__)           # la tarea pide Python 3.10+ y NLTK


## 3. Base de conocimiento, árboles y bot con memoria
- `BASE`: (intención, patrón regex, respuesta). **Gana la primera regla que coincide.**
- `ARBOL` / `HOJAS`: preguntas de lo general a lo específico para inscripción, becas, constancias y biblioteca.
- `OrientaBot`: clase que recuerda el árbol, el nodo y un mensaje pendiente.

In [ ]:
import re                                   # expresiones regulares: los «patrones» de cada regla
import unicodedata                          # quitar acentos al normalizar
from nltk.chat.util import Chat, reflections   # motor de reglas de NLTK (heredero de ELIZA, 1966)

# =====================================================================
# 1. BASE DE CONOCIMIENTO — (intención, patrón regex, respuesta)
# =====================================================================
# ⚠️ EL ORDEN IMPORTA: gana la PRIMERA regla cuyo patrón coincide.
#    Por eso «desconocida» (que acepta todo con '.*') va al final.
BASE = [
    ('saludo', r'^(hola|buenos dias|buenas tardes|buenas noches|hey)\b.*',
     '¡Hola! Soy OrientaBot. Consulta sobre inscripciones, horarios, becas, constancias o biblioteca.'),
    ('inscripcion', r'.*\b(inscripcion|inscripciones|inscribirme|reinscripcion)\b.*',
     'Consulta la convocatoria vigente y contacta a Control Escolar para conocer el proceso de inscripción '
     'o reinscripción. Verifica allí fechas y requisitos oficiales.'),
    ('horarios', r'.*\b(horario|horarios|clases)\b.*',
     'Consulta tu horario en el sistema escolar institucional. Si existe algún problema con tu grupo o '
     'materia, contacta a Control Escolar.'),
    ('becas', r'.*\b(beca|becas|apoyo economico)\b.*',
     'Consulta las convocatorias institucionales vigentes. El área de Becas puede orientarte sobre '
     'requisitos, documentos y fechas.'),
    ('constancias', r'.*\b(constancia|constancias|comprobante de estudios)\b.*',
     'Solicita información a Control Escolar sobre el procedimiento, documentos necesarios y tiempo de '
     'entrega de constancias.'),
    ('control_escolar', r'.*\b(control escolar|servicios escolares)\b.*',
     'Control Escolar orienta sobre trámites académicos. Consulta sus medios de contacto en el sitio '
     'oficial de tu institución.'),
    # --- RETO de la clase: nueva intención, ANTES de «desconocida» ---
    # Respuesta general que remite al área: no inventamos horarios ni requisitos.
    ('biblioteca', r'.*\b(biblioteca|libros|prestamo)\b.*',
     'La Biblioteca orienta sobre préstamos, renovaciones y consulta de libros. Revisa sus horarios y '
     'requisitos en el sitio oficial de tu institución o acude al mostrador de préstamo.'),
    ('ayuda', r'.*\b(ayuda|menu|opciones|que puedes hacer)\b.*',
     'Temas disponibles: inscripciones, horarios, becas, constancias, Control Escolar y biblioteca. '
     'Escribe salir para terminar.'),
    ('agradecimiento', r'.*\bgracias\b.*', '¡De nada! ¿Necesitas orientación sobre otro tema?'),
    ('despedida', r'^(salir|adios|hasta luego)[.!?]*$', '¡Hasta luego! Gracias por consultar OrientaBot.'),
    ('vacio', r'^$', 'Escribe una consulta para poder orientarte.'),
    # Red de seguridad (fallback): '.*' coincide con TODO, así el bot nunca se queda callado.
    ('desconocida', r'.*',
     'No identifiqué tu consulta. Intenta mencionar inscripciones, horarios, becas, constancias, '
     'Control Escolar o biblioteca.'),
]

# Chat de NLTK recibe pares (patrón, [respuestas]); si la lista trae varias, elige una al azar.
# Comprensión de lista: de cada tupla de 3 tomamos solo patrón y respuesta ('_' = «no me importa»).
chatbot = Chat([(patron, [respuesta]) for _, patron, respuesta in BASE], reflections)

# =====================================================================
# 2. FUNCIONES
# =====================================================================


def normalizar(texto):
    """Limpia el texto antes de compararlo: '  ¿Cómo  ME Inscribo? ' → '¿como me inscribo?'.

    Así «Inscripción», «inscripcion» e «INSCRIPCIÓN» caen en la misma regla.
    """
    texto = unicodedata.normalize('NFD', texto.lower().strip())             # minúsculas, sin espacios en orillas, separa acentos
    texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')    # quita los acentos
    return re.sub(r'\s+', ' ', texto)                                        # varios espacios seguidos → uno


def responder(texto):
    """Recibe lo que escribió el usuario y regresa {'intencion': …, 'respuesta': …}.

    La intención la calculamos nosotros (para medir); la respuesta la da NLTK.
    Coinciden porque ambos recorren los MISMOS patrones en el MISMO orden.
    """
    limpio = normalizar(texto)
    # next(generador): devuelve el PRIMER elemento que cumple la condición → «gana la primera regla».
    # Nunca falla porque «desconocida» ('.*') siempre coincide.
    intencion = next(nombre for nombre, patron, _ in BASE if re.match(patron, limpio, re.IGNORECASE))
    return {'intencion': intencion, 'respuesta': chatbot.respond(limpio)}


# =====================================================================
# 3. ÁRBOLES DE DECISIÓN — de lo general a lo específico
# =====================================================================
#   inscripcion ─┬─ nuevo_ingreso ─┬─ documentos         becas ─┬─ perdida (hoja)
#                │                 └─ fechas                    ├─ renovar ─┬─ documentos
#                └─ reinscripcion ─┬─ pagos                     │           └─ promedio
#                                  └─ materias                  └─ solicitar ─┬─ requisitos
#                                                                              └─ fechas
#   constancias ─┬─ calificaciones ─┬─ trámite           biblioteca ─┬─ perdida   (hoja)
#                │                  └─ costo y tiempo                ├─ renovacion (hoja)
#                └─ estudios ───────┬─ trámite                       └─ prestamo  (hoja)
#                                   └─ costo y tiempo
#
# Cada NODO hace una pregunta y tiene opciones: (patrón, a dónde lleva).
# Si «a dónde lleva» es otro nodo, seguimos bajando; si está en HOJAS, respondemos y salimos.
# La RAÍZ de cada árbol se llama igual que su intención en BASE: así sabemos por dónde entrar.
# ⚠️ Dentro de cada nodo también importa el orden: «renovar el préstamo» debe caer en renovación,
#    por eso renovación va antes que préstamo.
ARBOL = {
    # --- Inscripción ---
    'inscripcion': {
        'pregunta': '¿Eres de nuevo ingreso o vas a reinscribirte?',
        'opciones': [(r'\b(nuevo|primer|primera vez|aspirante|ingresar)\b', 'nuevo_ingreso'),
                     (r'\b(reinscri\w*|ya estudio|siguiente semestre)\b', 'reinscripcion')],
    },
    'nuevo_ingreso': {
        'pregunta': 'Nuevo ingreso. ¿Quieres saber de documentos o de fechas?',
        'opciones': [(r'\b(documento\w*|requisito\w*|papeles)\b', 'ni_documentos'),
                     (r'\b(fecha\w*|cuando|convocatoria)\b', 'ni_fechas')],
    },
    'reinscripcion': {
        'pregunta': 'Reinscripción. ¿Tu duda es sobre pagos o sobre materias?',
        'opciones': [(r'\b(pago\w*|pagar|costo\w*|cuanto)\b', 're_pagos'),
                     (r'\b(materia\w*|carga|grupo\w*)\b', 're_materias')],
    },
    # --- Becas ---
    'becas': {
        'pregunta': '¿Quieres solicitar una beca, renovar la que tienes, o la perdiste?',
        'opciones': [(r'\b(perdi\w*|quitaron|cancel\w*|suspend\w*)\b', 'be_perdida'),
                     (r'\b(renov\w*|mantener|conservar)\b', 'becas_renovar'),
                     (r'\b(solicit\w*|pedir|nueva|necesito|conseguir|aplicar|quiero)\b', 'becas_solicitar')],
    },
    'becas_renovar': {
        'pregunta': 'Renovación. ¿Tu duda es sobre documentos o sobre el promedio que piden?',
        'opciones': [(r'\b(documento\w*|papeles|requisito\w*)\b', 'be_ren_documentos'),
                     (r'\b(promedio|calificacion\w*|reprobe|reprobada)\b', 'be_ren_promedio')],
    },
    'becas_solicitar': {
        'pregunta': 'Solicitud nueva. ¿Quieres conocer los requisitos o las fechas de la convocatoria?',
        'opciones': [(r'\b(requisito\w*|documento\w*|promedio|papeles)\b', 'be_requisitos'),
                     (r'\b(fecha\w*|cuando|convocatoria|plazo)\b', 'be_fechas')],
    },
    # --- Constancias ---
    'constancias': {
        'pregunta': '¿La necesitas de estudios (simple) o con calificaciones?',
        'opciones': [(r'\b(calificacion\w*|kardex|promedio|historial)\b', 'const_calificaciones'),
                     (r'\b(estudios|simple|normal|inscrito|imss|trabajo)\b', 'const_estudios')],
    },
    'const_calificaciones': {
        'pregunta': 'Con calificaciones. ¿Quieres saber cómo pedirla o cuánto cuesta y tarda?',
        'opciones': [(r'\b(costo|cuesta|precio|pagar|tarda|tiempo|dias)\b', 'co_cal_costo'),
                     (r'\b(como|pedir\w*|solicit\w*|donde|tramit\w*|sacar\w*)\b', 'co_cal_tramite')],
    },
    'const_estudios': {
        'pregunta': 'De estudios. ¿Quieres saber cómo pedirla o cuánto cuesta y tarda?',
        'opciones': [(r'\b(costo|cuesta|precio|pagar|tarda|tiempo|dias)\b', 'co_est_costo'),
                     (r'\b(como|pedir\w*|solicit\w*|donde|tramit\w*|sacar\w*)\b', 'co_est_tramite')],
    },
    # --- Biblioteca ---
    'biblioteca': {
        'pregunta': '¿Es sobre un préstamo, una renovación, o un libro perdido o dañado?',
        'opciones': [(r'\b(perdi\w*|perdido|dan\w*|maltrat\w*|rompi\w*)\b', 'bi_perdida'),
                     (r'\b(renov\w*|extender|mas tiempo)\b', 'bi_renovacion'),
                     (r'\b(prestamo|pedir|llevar\w*|sacar)\b', 'bi_prestamo')],
    },
}

# Hojas = respuestas finales. Generales y remitiendo al área: no inventamos datos oficiales.
HOJAS = {
    'ni_documentos': 'Los documentos de nuevo ingreso vienen en la convocatoria vigente. Control Escolar '
                     'te confirma la lista oficial.',
    'ni_fechas': 'Las fechas de nuevo ingreso se publican en la convocatoria del sitio oficial.',
    're_pagos': 'El monto y la forma de pago de la reinscripción los informa Control Escolar o el portal '
                'de pagos institucional.',
    're_materias': 'La selección de materias se hace en el sistema escolar. Si tienes adeudos o empalmes, '
                   'contacta a Control Escolar.',
    'be_perdida': 'Si perdiste tu beca, acude al área de Becas: te explican la causa y si puedes '
                  'apelar o volver a solicitarla en la siguiente convocatoria.',
    'be_ren_documentos': 'Los documentos para renovar vienen en la convocatoria de renovación. El área de '
                         'Becas te confirma la lista.',
    'be_ren_promedio': 'El promedio mínimo para conservar la beca lo fija el reglamento de Becas. Si estás '
                       'en riesgo, consúltalo con el área antes del cierre de periodo.',
    'be_requisitos': 'Los requisitos de cada beca vienen en su convocatoria. El área de Becas te orienta '
                     'sobre cuál te conviene.',
    'be_fechas': 'Las convocatorias de becas se publican en el sitio oficial; revisa ahí los plazos.',
    'co_cal_tramite': 'La constancia con calificaciones se solicita en Control Escolar o en el portal de '
                      'trámites, si tu institución lo tiene.',
    'co_cal_costo': 'El costo y el tiempo de entrega de la constancia con calificaciones los informa '
                    'Control Escolar.',
    'co_est_tramite': 'La constancia de estudios se solicita en Control Escolar o en el portal de trámites, '
                      'si tu institución lo tiene.',
    'co_est_costo': 'El costo y el tiempo de entrega de la constancia de estudios los informa Control Escolar.',
    'bi_perdida': 'Si perdiste o dañaste un libro, repórtalo en el mostrador de la Biblioteca: te indican '
                  'cómo reponerlo.',
    'bi_renovacion': 'Las renovaciones se hacen en el mostrador de la Biblioteca o en su catálogo en línea, '
                     'antes de la fecha de vencimiento.',
    'bi_prestamo': 'Para un préstamo acude al mostrador de la Biblioteca con tu credencial vigente.',
}


class OrientaBot:
    """Bot CON memoria: recuerda en qué árbol y en qué rama va la conversación.

    Una clase junta datos (el estado) y funciones (métodos) que los usan.
    Cada estudiante tendría su propio objeto: OrientaBot() → conversación nueva.
    """

    def __init__(self):
        # __init__ corre al crear el objeto. «self» es el propio objeto.
        self.tema = None          # árbol actual: 'inscripcion', 'becas'… (None = fuera de los árboles)
        self.nodo = None          # rama actual dentro de ese árbol
        self.pendiente = None     # mensaje que no supimos ubicar aún (p. ej. «¿qué documentos?»)

    def _bajar(self, limpio):
        """Intenta avanzar por el árbol con este texto. Regresa la hoja si llegó a una, o None.

        El guion bajo inicial (_bajar) es convención: «método interno, no lo llames desde fuera».
        """
        avanzo = True
        # Mientras estemos en un nodo y el texto encaje en alguna opción, seguimos bajando.
        # Así «me reinscribo y quiero saber de pagos» llega a la hoja en UN solo mensaje.
        while self.nodo and avanzo:
            avanzo = False
            for patron, destino in ARBOL[self.nodo]['opciones']:
                if re.search(patron, limpio):        # search busca en cualquier parte (match solo al inicio)
                    if destino in HOJAS:
                        self.nodo = None             # llegamos al final: salimos del árbol
                        return destino
                    self.nodo, avanzo = destino, True
                    break
        return None

    def _salida(self, hoja=None, prefijo=''):
        """Arma la respuesta: la hoja si llegamos a una, o la pregunta del nodo actual."""
        tema = self.tema
        if hoja:
            self.tema = self.pendiente = None        # conversación del árbol terminada
            return {'intencion': tema, 'nodo': hoja, 'respuesta': HOJAS[hoja]}
        return {'intencion': tema, 'nodo': self.nodo, 'respuesta': prefijo + ARBOL[self.nodo]['pregunta']}

    def _entrar(self, tema, limpio):
        """Entra a un árbol por su raíz e intenta bajar con el mismo mensaje."""
        self.tema, self.nodo, self.pendiente = tema, tema, None
        hoja = self._bajar(limpio)
        if not hoja:
            # Guardamos el mensaje: puede traer un detalle que sirva más abajo.
            # «¿Cuándo abren las inscripciones?» → al llegar a nuevo_ingreso, «cuándo» elige fechas.
            self.pendiente = limpio
        return self._salida(hoja, prefijo='Con gusto. ')

    def responder(self, texto):
        """Como responder() de arriba, pero usando y actualizando el estado."""
        limpio = normalizar(texto)
        global_ = responder(texto)                   # lo que diría el bot sin memoria
        intencion = global_['intencion']

        # Caso A: estamos dentro de un árbol.
        if self.nodo:
            antes = self.nodo                        # para saber si este mensaje nos hizo avanzar
            hoja = self._bajar(limpio)
            # Si avanzamos y había algo pendiente, lo usamos ahora:
            # «quiero inscribirme» → «¿qué documentos?» → «nuevo ingreso» ⇒ responde documentos.
            if self.pendiente and not hoja and self.nodo != antes:
                hoja = self._bajar(self.pendiente)
            if hoja:
                return self._salida(hoja)
            if self.nodo != antes:                   # avanzó a otra rama → hacemos su pregunta
                return self._salida()
            # No avanzó: ¿cambió de tema?
            if intencion in ARBOL and intencion != self.tema:        # a otro árbol → entramos a ese
                return self._entrar(intencion, limpio)
            if intencion not in ('desconocida', self.tema):          # a un tema simple → salimos
                self.tema = self.nodo = self.pendiente = None
                return {**global_, 'nodo': None}
            # Ni avanzó ni cambió de tema: guardamos su mensaje y repetimos la pregunta.
            self.pendiente = limpio
            return self._salida(prefijo='Para orientarte mejor: ')

        # Caso B: fuera de los árboles. Si el tema tiene árbol, entramos por su raíz.
        if intencion in ARBOL:
            return self._entrar(intencion, limpio)

        # Cualquier otro tema: respuesta normal de una sola pregunta.
        return {**global_, 'nodo': None}


# =====================================================================
# 4. CHAT EN LA TERMINAL (solo si ejecutas este archivo directo)
# =====================================================================

## 4. Pseudoestudiantes: 200 preguntas realistas
10 perfiles × 20 preguntas. Mezcla **no uniforme** de temas y ~1 de cada 3 preguntas «sucia»
(dedazos de tecla vecina, letras cambiadas, sin acentos, abreviaturas). La intención esperada es la que el
estudiante **quiere**, aunque escriba mal. Semilla fija: siempre salen las mismas.

In [ ]:
# --- Importaciones: solo la biblioteca estándar, nada que instalar ---
import csv                # escribir el resultado como tabla .csv (se abre en Excel)
import random             # azar controlado: elegir plantillas, errores, perfiles
import unicodedata        # separar letras de sus acentos (á → a + ´)
from pathlib import Path  # manejar rutas de archivos sin pelear con / o \

# =====================================================================
# 1. CONFIGURACIÓN — lo único que deberías tocar para cambiar el pool
# =====================================================================

# Semilla: con el mismo número, random da SIEMPRE la misma secuencia.
# Así el pool es reproducible: tú y el profe obtienen las mismas 200 preguntas.
SEMILLA = 42
PREGUNTAS_POR_ESTUDIANTE = 20

# Diccionario tema → porcentaje del pool (suma 100).
# No es uniforme a propósito: en la vida real casi todos preguntan por inscripción.
MEZCLA = {
    'inscripcion': 22, 'horarios': 15, 'constancias': 12, 'becas': 10,
    'desconocida': 12, 'saludo': 4, 'agradecimiento': 3, 'despedida': 3,
    'biblioteca': 6, 'control_escolar': 5, 'ayuda': 3, 'dos_temas': 5,
}

# Lista de tuplas (nombre, cómo escribe, probabilidad de escribir «sucio»).
# 0.05 = se equivoca 1 de cada 20 veces; 0.70 = 7 de cada 10.
# El promedio de la columna da ~0.33: 1 de cada 3 preguntas sucia.
PERFILES = [
    ('Mariana', 'formal, revisa antes de enviar', 0.05),
    ('Diego', 'cuidadoso, frases completas', 0.10),
    ('Sofía', 'aspirante de nuevo ingreso, pregunta todo', 0.15),
    ('Andrés', 'directo, pocas palabras', 0.20),
    ('Valeria', 'escribe rápido entre clases', 0.30),
    ('Luis', 'desde el celular, sin acentos', 0.35),
    ('Fernanda', 'informal, usa abreviaturas', 0.40),
    ('Jorge', 'trabaja y estudia, siempre con prisa', 0.50),
    ('Camila', 'escribe con el pulgar, muchos dedazos', 0.55),
    ('Kevin', 'no revisa nada, puro dedazo', 0.70),
]

# Diccionario intención → lista de frases «limpias».
# La CLAVE es la etiqueta correcta (lo que el estudiante quiere);
# el bot tendrá que adivinarla solo con el texto.
PLANTILLAS = {
    'inscripcion': [
        '¿Cómo puedo inscribirme?', '¿Cuándo abren las inscripciones?',
        '¿Qué necesito para la reinscripción?', 'Quiero inscribirme al siguiente semestre',
        '¿Hasta cuándo hay inscripciones?', 'No sé cómo hacer mi reinscripción',
        '¿Dónde hago la inscripción?', 'Me quiero inscribir, ¿qué hago?',
        '¿Qué documentos piden para la inscripción?', '¿La reinscripción es en línea?',
        'Perdí la fecha de inscripción, ¿todavía puedo?', '¿Cuánto cuesta la inscripción?',
    ],
    'horarios': [
        '¿Dónde veo mi horario?', '¿Ya salieron los horarios?', '¿A qué hora son mis clases?',
        'No me aparece mi horario', 'Quiero cambiar mi horario', '¿Cuándo empiezan las clases?',
        'Tengo un empalme en mi horario', '¿Dónde consulto los horarios del grupo?',
        '¿Las clases son presenciales?', 'Mi horario está mal',
    ],
    'constancias': [
        'Necesito una constancia de estudios', '¿Cómo saco una constancia?',
        '¿Cuánto tarda la constancia?', 'Quiero un comprobante de estudios',
        '¿Dónde pido constancias?', 'Ocupo una constancia para el IMSS',
        '¿La constancia tiene costo?', 'Necesito constancia con calificaciones',
    ],
    'becas': [
        '¿Hay becas?', '¿Cómo solicito una beca?', 'Necesito apoyo económico',
        '¿Cuándo sale la convocatoria de becas?', '¿Qué promedio piden para la beca?',
        'Me quitaron la beca, ¿qué hago?', '¿Puedo renovar mi beca?',
        'No tengo beca pero la necesito',
    ],
    'biblioteca': [
        'Quiero pedir libros', '¿A qué hora abre la biblioteca?', '¿Cómo pido un préstamo?',
        '¿Puedo renovar el préstamo de un libro?', '¿Dónde está la biblioteca?',
        'Perdí un libro de la biblioteca',
    ],
    'control_escolar': [
        '¿Dónde está Control Escolar?', '¿Cuál es el teléfono de Servicios Escolares?',
        '¿A qué hora atiende Control Escolar?', 'Necesito hablar con Control Escolar',
        '¿Control escolar tiene correo?',
    ],
    'ayuda': ['¿Qué puedes hacer?', 'Ayuda', 'Menú', '¿Qué opciones tienes?', 'No sé qué preguntar'],
    'saludo': ['Hola', 'Buenos días', 'Buenas tardes', 'Hey', 'Buenas noches', 'Hola, buen día'],
    'agradecimiento': ['Gracias', 'Muchas gracias', 'Gracias por la info', 'Ok, gracias'],
    'despedida': ['Adiós', 'Hasta luego', 'Salir', 'Bye', 'Nos vemos'],
    # «desconocida» = fuera de alcance. Lo CORRECTO aquí es que el bot diga «no sé».
    'desconocida': [
        '¿Cómo pago la colegiatura?', 'Olvidé mi contraseña del portal', '¿Qué clima hará mañana?',
        '¿Hay estacionamiento?', '¿Dónde está la cafetería?', '¿Cómo me doy de baja?',
        'No puedo entrar a la plataforma', '¿Cuándo son vacaciones?', '¿Hay servicio médico?',
        '¿Venden uniformes?', '¿Cuándo es la graduación?', 'Quiero cambiarme de carrera',
    ],
}

# Frases con dos temas: (texto, intención principal).
# Tomamos como correcta la que el estudiante quiere de verdad (en la negación, la segunda).
DOS_TEMAS = [
    ('Quiero saber de becas y horarios', 'becas'),
    ('Necesito una constancia y también inscribirme', 'constancias'),
    ('¿Dónde veo mi horario y dónde está Control Escolar?', 'horarios'),
    ('Ocupo inscribirme y saber si hay becas', 'inscripcion'),
    ('No quiero beca, quiero una constancia', 'constancias'),
    ('¿La biblioteca tiene horario en vacaciones?', 'biblioteca'),
]

# Muletillas al inicio y al final. Las '' repetidas hacen que «nada» salga más seguido:
# random.choice elige con la misma probabilidad cada elemento de la lista.
ABRIDORES = ['', '', '', 'Oye, ', 'Disculpa, ', 'Una pregunta, ', 'Hola, ']
CIERRES = ['', '', '', ' por favor', ' gracias', '!!']

# Cómo abrevia la gente en el chat: palabra completa → abreviatura.
ABREVIATURAS = {'que': 'q', 'por favor': 'xfa', 'para': 'pa', 'porque': 'xq',
                'también': 'tmb', 'gracias': 'grax', 'quiero': 'kiero', 'dónde': 'dnd'}

# Mapa del teclado QWERTY en español: cada letra → las letras que la rodean.
# Así un dedazo es realista: «c» se vuelve «v» (vecina), no «z» (lejana).
VECINAS = {
    'q': 'wa', 'w': 'qes', 'e': 'wrd', 'r': 'etf', 't': 'ryg', 'y': 'tuh', 'u': 'yij',
    'i': 'uok', 'o': 'ipl', 'p': 'oñ', 'a': 'qsz', 's': 'adwx', 'd': 'sfec', 'f': 'dgrv',
    'g': 'fhtb', 'h': 'gjyn', 'j': 'hkum', 'k': 'jli', 'l': 'kñ', 'ñ': 'lp', 'z': 'xa',
    'x': 'zcs', 'c': 'xvd', 'v': 'cbf', 'b': 'vng', 'n': 'bmh', 'm': 'nj',
}

# =====================================================================
# 2. FUNCIONES — cada una hace UNA cosa
# =====================================================================


def sin_acentos(texto):
    """Quita acentos: 'inscripción' → 'inscripcion'.

    Truco: NFD separa cada letra de su acento ('ó' → 'o' + '´').
    Luego descartamos los caracteres de categoría 'Mn' (marcas = acentos).
    """
    texto = unicodedata.normalize('NFD', texto)                            # separa letra y acento
    return ''.join(c for c in texto if unicodedata.category(c) != 'Mn')   # se queda solo con letras


def dedazo(palabra, rng):
    """Mete UN error de tecleo en una palabra y dice cuál fue.

    Regresa una tupla (palabra_con_error, tipo_de_error), por ejemplo
    ('horaruo', 'tecla_vecina'). Python permite regresar varios valores así.
    """
    # Posición del error: nunca la primera ni la última letra
    # (la gente casi siempre atina al inicio de la palabra).
    i = rng.randrange(1, len(palabra) - 1)
    tipo = rng.choice(['tecla_vecina', 'letras_cambiadas', 'falta_letra', 'sobra_letra'])

    # Rebanadas (slicing): palabra[:i] = todo ANTES de i, palabra[i+1:] = todo DESPUÉS.
    # Reemplazar la letra i es pegar: antes + letra_nueva + después.
    if tipo == 'tecla_vecina' and palabra[i].lower() in VECINAS:
        return palabra[:i] + rng.choice(VECINAS[palabra[i].lower()]) + palabra[i + 1:], tipo   # horario → horaruo
    # Intercambiar i e i+1; si son iguales («ss») el cambio sería invisible, así que lo saltamos.
    if tipo == 'letras_cambiadas' and palabra[i] != palabra[i + 1]:
        return palabra[:i] + palabra[i + 1] + palabra[i] + palabra[i + 2:], tipo               # becas → beacs
    if tipo == 'falta_letra':
        return palabra[:i] + palabra[i + 1:], tipo                                             # constancia → constacia
    # Caso por defecto (y respaldo si un caso de arriba no aplicó): duplicar la letra i.
    return palabra[:i] + palabra[i] + palabra[i:], 'sobra_letra'                               # horario → horrario


def ensuciar(texto, rng):
    """Convierte una frase limpia en una escrita «con prisa».

    Aplica capas de error al azar y regresa (texto_sucio, lista_de_errores).
    La lista sirve después para saber QUÉ tipo de error tumbó al bot.
    """
    errores = []

    # Capa 1 (60 %): todo en minúsculas, sin acentos y sin '¿' — como en el celular.
    if rng.random() < 0.6:                    # random() da un número entre 0 y 1
        texto = sin_acentos(texto).lower().replace('¿', '')
        errores.append('sin_acentos')

    # Capa 2 (40 %): cambia UNA palabra por su abreviatura (q, xfa, pa…).
    if rng.random() < 0.4:
        for largo, corto in ABREVIATURAS.items():
            # Los espacios alrededor evitan falsos positivos: 'que' NO debe pegar dentro de 'quedan'.
            if f' {largo} ' in f' {texto} ':
                texto = texto.replace(largo, corto)
                errores.append('abreviatura')
                break                         # basta con una abreviatura por frase

    # Capa 3 (siempre): 1 o 2 dedazos en palabras de 4+ letras.
    palabras = texto.split()                  # 'hola que tal' → ['hola', 'que', 'tal']
    # Comprensión de lista: guardamos los ÍNDICES de las palabras largas (sin contar signos).
    largas = [i for i, p in enumerate(palabras) if len(p.strip('¿?,.!')) >= 4]
    # [1, 1, 2]: dos tercios de las veces 1 dedazo, un tercio 2. min() evita pedir más de las que hay.
    for i in rng.sample(largas, k=min(len(largas), rng.choice([1, 1, 2]))):
        palabras[i], tipo = dedazo(palabras[i], rng)   # desempaquetado: la tupla se reparte en 2 variables
        errores.append(tipo)
    return ' '.join(palabras), errores        # volvemos a unir la lista en una frase


def generar_pool(semilla=SEMILLA):
    """Arma las 200 preguntas: a cada estudiante le toca su porción de temas.

    Regresa una lista de diccionarios; cada diccionario es una fila del CSV.
    """
    # Generador de azar PROPIO con semilla: no depende de nada externo, siempre igual.
    rng = random.Random(semilla)

    # Paso 1: convertir porcentajes en una «bolsa» de temas.
    # 22 % de 200 = 44 → la lista lleva 'inscripcion' 44 veces, y así con cada tema.
    total = len(PERFILES) * PREGUNTAS_POR_ESTUDIANTE
    temas = [tema for tema, pct in MEZCLA.items() for _ in range(round(total * pct / 100))]
    rng.shuffle(temas)                        # revolvemos la bolsa (modifica la lista en su lugar)

    pool = []
    # enumerate da (número, elemento); la tupla del perfil se desempaqueta en 3 variables.
    for n, (nombre, estilo, tasa_sucio) in enumerate(PERFILES):
        # Rebanada de 20 temas para este estudiante: el 0 toma [0:20], el 1 toma [20:40]…
        for tema in temas[n * PREGUNTAS_POR_ESTUDIANTE:(n + 1) * PREGUNTAS_POR_ESTUDIANTE]:

            # Paso 2: elegir la frase limpia y su respuesta correcta.
            if tema == 'dos_temas':
                texto, esperada = rng.choice(DOS_TEMAS)            # la tupla trae su propia etiqueta
            else:
                texto, esperada = rng.choice(PLANTILLAS[tema]), tema

            # Paso 3: muletillas («Oye, …», «… por favor»). No a saludos ni despedidas:
            # «Oye, hola gracias» no lo escribe nadie.
            if tema not in ('saludo', 'agradecimiento', 'despedida', 'ayuda'):
                abridor = rng.choice(ABRIDORES)
                if abridor:
                    # Tras la muletilla la frase sigue en minúscula: «Oye, ¿cómo…».
                    # k salta el '¿' para bajar la primera LETRA, no el signo.
                    k = 1 if texto.startswith('¿') else 0
                    texto = abridor + texto[:k] + texto[k].lower() + texto[k + 1:]
                texto += rng.choice(CIERRES)

            # Paso 4: ¿este estudiante escribe sucio esta vez? Depende de su perfil.
            errores = []
            if rng.random() < tasa_sucio:
                texto, errores = ensuciar(texto, rng)

            # Paso 5: guardar la fila. bool([]) es False y bool(['x']) es True.
            pool.append({'estudiante': nombre, 'estilo': estilo, 'tipo': tema, 'pregunta': texto,
                         'esperada': esperada, 'sucia': bool(errores), 'errores': '+'.join(errores)})
    return pool


pool = generar_pool()
import pandas as pd
pd.DataFrame(pool).sample(12, random_state=1)[['estudiante', 'pregunta', 'esperada', 'errores']]

## 5. Evaluación de una sola pregunta (métricas del tema 4)
**Precisión de intención** y **tasa de fallback** sobre las 200 preguntas, separadas por limpias/sucias,
intención y estudiante.

In [ ]:
import csv
from collections import Counter, defaultdict   # Counter cuenta cosas; defaultdict crea llaves solas
from pathlib import Path


CARPETA = Path('.')        # carpeta de este archivo


def pct(parte, total):
    """Porcentaje con formato bonito: pct(3, 4) → ' 75%'. Evita dividir entre cero."""
    return f'{parte / total:4.0%}' if total else '   —'


def evaluar(pool):
    """Le pasa cada pregunta al bot y anota si acertó. Regresa la lista de filas con resultado."""
    resultados = []
    for fila in pool:
        salida = responder(fila['pregunta'])
        # {**fila, …}: copia todas las columnas de la fila y le agrega las nuevas.
        resultados.append({**fila, 'detectada': salida['intencion'],
                           'correcto': salida['intencion'] == fila['esperada']})
    return resultados


def reporte(resultados):
    """Imprime las métricas, de lo general a lo particular."""
    total = len(resultados)
    aciertos = sum(r['correcto'] for r in resultados)                  # True cuenta como 1
    fallback = sum(r['detectada'] == 'desconocida' for r in resultados)
    print(f'\n== OrientaBot vs. {total} preguntas ==')
    print(f'Precisión de intención: {aciertos}/{total} = {pct(aciertos, total)}')
    print(f'Tasa de fallback:       {fallback}/{total} = {pct(fallback, total)}')

    # --- ¿Escribir mal le pega al bot? Comparamos limpias contra sucias ---
    print('\n-- Limpias vs. sucias --')
    for sucia in (False, True):
        grupo = [r for r in resultados if r['sucia'] == sucia]
        ok = sum(r['correcto'] for r in grupo)
        print(f"{'sucias ' if sucia else 'limpias'}: {ok}/{len(grupo)} = {pct(ok, len(grupo))}")

    # --- Precisión por intención esperada ---
    # defaultdict(list): si la llave no existe, la crea con una lista vacía (sin if).
    por_intencion = defaultdict(list)
    for r in resultados:
        por_intencion[r['esperada']].append(r['correcto'])
    print('\n-- Por intención --')
    # sorted(..., key=…): ordena de la intención con más preguntas a la de menos.
    for intencion, lista in sorted(por_intencion.items(), key=lambda par: -len(par[1])):
        print(f'{intencion:16} {sum(lista):3}/{len(lista):<3} {pct(sum(lista), len(lista))}')

    # --- Por estudiante: ¿Kevin la pasa peor que Mariana? ---
    por_estudiante = defaultdict(list)
    for r in resultados:
        por_estudiante[r['estudiante']].append(r['correcto'])
    print('\n-- Por estudiante --')
    for nombre, lista in por_estudiante.items():   # los dicts conservan el orden de inserción
        print(f'{nombre:9} {pct(sum(lista), len(lista))}')

    # --- Confusiones más comunes: qué se esperaba → qué detectó ---
    confusiones = Counter((r['esperada'], r['detectada']) for r in resultados if not r['correcto'])
    print('\n-- Confusiones más comunes (esperada → detectada) --')
    for (esperada, detectada), n in confusiones.most_common(8):
        print(f'{n:3} × {esperada} → {detectada}')


def guardar(resultados):
    """Guarda cada pregunta con su resultado; sirve de evidencia para el PDF."""
    salida = CARPETA / 'resultados.csv'
    with salida.open('w', newline='', encoding='utf-8') as f:
        escritor = csv.DictWriter(f, fieldnames=list(resultados[0]))
        escritor.writeheader()
        escritor.writerows(resultados)
    print(f'\nDetalle guardado en {salida.name}')


resultados = evaluar(pool)
reporte(resultados)

## 6. 21 conversaciones de varios turnos
**14 exitosas** (llegan a la respuesta final que buscaban) y **7 fallidas**, cada una por un motivo realista.
Lo esperado se declara **antes** de ejecutar; ✅ = se comportó como se declaró.

In [ ]:
from pathlib import Path


# Cada flujo es un diccionario:
#   estudiante → quién conversa (los perfiles de estudiantes.py)
#   resumen    → qué pasa, en una línea
#   turnos     → lo que escribe el estudiante, en orden
#   hojas      → respuestas finales que DEBE alcanzar ([] = debe fallar)
#   motivo     → solo en las fallidas: por qué se queda en el camino
EXITOSOS = [
    {'estudiante': 'Sofía', 'resumen': 'Pregunta documentos antes de decir que es de nuevo ingreso; el bot lo recuerda',
     'turnos': ['Hola', 'quiero inscribirme', '¿qué documentos necesito?', 'soy de nuevo ingreso', 'gracias'],
     'hojas': ['ni_documentos']},
    {'estudiante': 'Diego', 'resumen': 'La fecha va en la primera pregunta; se usa al aclarar que es nuevo',
     'turnos': ['Buenas tardes', '¿Cuándo abren las inscripciones?', 'es mi primera vez', 'Muchas gracias'],
     'hojas': ['ni_fechas']},
    {'estudiante': 'Andrés', 'resumen': 'Todo en un solo mensaje: baja dos niveles de golpe',
     'turnos': ['Me toca reinscripción y quiero saber de pagos', 'gracias', 'adiós'],
     'hojas': ['re_pagos']},
    {'estudiante': 'Valeria', 'resumen': 'Recorre el árbol paso a paso, un nivel por mensaje',
     'turnos': ['inscripción', 'ya estudio aquí, voy al siguiente semestre', 'es por las materias', 'ok gracias'],
     'hojas': ['re_materias']},
    {'estudiante': 'Mariana', 'resumen': 'Flujo formal completo: saludo, beca nueva, requisitos, cierre',
     'turnos': ['Hola', '¿Cómo solicito una beca?', 'Los requisitos, por favor', 'Gracias', 'Adiós'],
     'hojas': ['be_requisitos']},
    {'estudiante': 'Luis', 'resumen': 'Sin acentos ni signos, desde el celular',
     'turnos': ['necesito una beca', 'cuando sale la convocatoria', 'gracias'],
     'hojas': ['be_fechas']},
    {'estudiante': 'Jorge', 'resumen': 'Beca perdida en un mensaje y luego cambia a horarios',
     'turnos': ['Me quitaron la beca', '¿y dónde veo mi horario?', 'hasta luego'],
     'hojas': ['be_perdida']},
    {'estudiante': 'Camila', 'resumen': 'Renovación de beca: le preocupa el promedio',
     'turnos': ['quiero renovar mi beca', '¿qué promedio piden?', 'gracias'],
     'hojas': ['be_ren_promedio']},
    {'estudiante': 'Fernanda', 'resumen': 'Informal y con abreviaturas, pero las palabras clave pasan',
     'turnos': ['beca', 'renovarla', 'q papeles ocupo', 'grax'],
     'hojas': ['be_ren_documentos']},
    {'estudiante': 'Diego', 'resumen': 'Constancia con calificaciones: costo',
     'turnos': ['Necesito una constancia', 'con calificaciones', '¿cuánto cuesta?', 'Gracias'],
     'hojas': ['co_cal_costo']},
    {'estudiante': 'Sofía', 'resumen': 'El IMSS delata que es constancia de estudios',
     'turnos': ['Ocupo una constancia para el IMSS', '¿cómo la pido?'],
     'hojas': ['co_est_tramite']},
    {'estudiante': 'Andrés', 'resumen': 'Dos árboles seguidos: constancia y luego biblioteca',
     'turnos': ['constancia de estudios, ¿cuánto tarda?', 'también perdí un libro de la biblioteca', 'gracias'],
     'hojas': ['co_est_costo', 'bi_perdida']},
    {'estudiante': 'Valeria', 'resumen': 'Renovar un préstamo: gana renovación por el orden de las opciones',
     'turnos': ['¿Puedo renovar el préstamo de un libro?', 'gracias'],
     'hojas': ['bi_renovacion']},
    {'estudiante': 'Kevin', 'resumen': 'Empieza en inscripción y se cambia de árbol a biblioteca',
     'turnos': ['quiero inscribirme', 'mejor dime de la biblioteca', 'un prestamo', 'gracias'],
     'hojas': ['bi_prestamo']},
]

FALLIDOS = [
    {'estudiante': 'Kevin', 'motivo': 'Abandono',
     'resumen': 'Entra al árbol, contesta «ok» y se va sin elegir rama',
     'turnos': ['quiero inscribirme', 'ok'], 'hojas': []},
    {'estudiante': 'Camila', 'motivo': 'Dedazo dentro del árbol',
     'resumen': '«requsitos» no coincide con el patrón; el bot repite la pregunta hasta que ella se rinde',
     'turnos': ['necesito una beca', 'los requsitos', 'requsitos xfa', 'olvidalo'], 'hojas': []},
    {'estudiante': 'Jorge', 'motivo': 'Dedazo al entrar',
     'resumen': 'Ni «kiero inscirbirme» ni «inscirpcion» activan la regla: cae dos veces en el fallback',
     'turnos': ['kiero inscirbirme', 'inscirpcion', 'bye'], 'hojas': []},
    {'estudiante': 'Luis', 'motivo': 'El saludo se come la consulta',
     'resumen': '«Hola, necesito una constancia» cae en saludo; nunca entra al árbol',
     'turnos': ['Hola, necesito una constancia', 'ok'], 'hojas': []},
    {'estudiante': 'Fernanda', 'motivo': 'Respuesta fuera de las opciones',
     'resumen': 'Contesta con algo que el nodo no prevé (la visa) y no sabe nombrar el tipo',
     'turnos': ['constancias', 'la que piden para la visa', 'no se cual', 'ya vere'], 'hojas': []},
    {'estudiante': 'Mariana', 'motivo': 'Vocabulario no previsto',
     'resumen': '«Me quiero inscribir» no está en el patrón (solo «inscribirme»); se va al menú y desiste',
     'turnos': ['Me quiero inscribir', '¿Qué puedes hacer?', 'Hasta luego'], 'hojas': []},
    {'estudiante': 'Andrés', 'motivo': 'Pierde el contexto al cambiar de tema',
     'resumen': 'Iba a renovar su beca, pregunta su horario y al volver el bot ya olvidó «renovar»',
     'turnos': ['quiero renovar mi beca', '¿dónde veo mi horario?', '¿y lo de la beca?', 'adios'], 'hojas': []},
]


def correr(flujo):
    """Corre un flujo con un bot NUEVO (sin memoria previa). Regresa (transcripción, hojas alcanzadas)."""
    bot = OrientaBot()
    lineas, hojas = [], []
    for turno in flujo['turnos']:
        salida = bot.responder(turno)
        # Una hoja es una respuesta final: su nombre está en HOJAS (no es un nodo ni una intención simple).
        if salida['nodo'] in HOJAS:
            hojas.append(salida['nodo'])
        lineas.append(f"  {flujo['estudiante']}: {turno}")
        lineas.append(f"  OrientaBot: {salida['respuesta']}   [{salida['intencion']} · {salida['nodo']}]")
    return lineas, hojas


texto_flujos, ok = [], 0
for n, flujo in enumerate(EXITOSOS + FALLIDOS, 1):
    lineas, hojas = correr(flujo)
    cumple = hojas == flujo['hojas']
    ok += cumple
    tipo = 'EXITOSO' if flujo['hojas'] else f"FALLIDO · {flujo['motivo']}"
    texto_flujos += [f"=== Flujo {n:02d} · {tipo} · {'✅' if cumple else '❌ REVISAR'} ===", f"  {flujo['resumen']}", *lineas, '']
texto_flujos.append(f'{ok}/{len(EXITOSOS + FALLIDOS)} flujos se comportan como se declaró.')
print('\n'.join(texto_flujos))

## 7. Conversación manual (opcional)
Cambia `ACTIVAR_CHAT` a `True` y ejecuta. Escribe `salir` para terminar.

In [ ]:
ACTIVAR_CHAT = False
if ACTIVAR_CHAT:
    bot = OrientaBot()
    print('OrientaBot: ¡Hola! Escribe tu consulta o salir.')
    while True:
        salida = bot.responder(input('Tú: '))
        print('OrientaBot:', salida['respuesta'], f"  [{salida['intencion']} · {salida['nodo']}]")
        if salida['intencion'] == 'despedida':
            break
else:
    print('Activa ACTIVAR_CHAT para conversar manualmente.')

## 8. Exportación de evidencias
CSV con las 200 preguntas y su resultado + TXT con las 21 conversaciones, en un ZIP.

In [ ]:
import csv, zipfile
from datetime import datetime, timezone
import nltk

carpeta = Path('evidencias_orientabot')
carpeta.mkdir(exist_ok=True)
with (carpeta / 'resultados_200.csv').open('w', newline='', encoding='utf-8-sig') as f:
    escritor = csv.DictWriter(f, fieldnames=list(resultados[0]))
    escritor.writeheader()
    escritor.writerows(resultados)
encabezado = [f'Ejecución UTC: {datetime.now(timezone.utc).isoformat()}',
              f'Python: {sys.version.split()[0]} | NLTK: {nltk.__version__}', '']
(carpeta / 'flujos_21.txt').write_text('\n'.join(encabezado + texto_flujos), encoding='utf-8')
with zipfile.ZipFile('Evidencias_OrientaBot.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for archivo in carpeta.iterdir():
        z.write(archivo, arcname=archivo.name)
print('Evidencias generadas: Evidencias_OrientaBot.zip')
DESCARGAR = False  # True para descargar el ZIP en Colab
if DESCARGAR:
    from google.colab import files
    files.download('Evidencias_OrientaBot.zip')

## 9. Resultados, limitaciones y siguientes etapas
- **Precisión de intención (200 preguntas):** 65 % · limpias 77 % · sucias 43 % · fallback 25 %.
- **Conversaciones:** 21/21 se comportan como se declaró (14 exitosas, 7 fallidas por motivos conocidos).
- **Limitaciones:** coincidencia exacta de palabras (dedazos y sinónimos fallan), prioridad por orden
  (el saludo se come la consulta), contexto limitado a un árbol a la vez, sin manejo de negaciones.
  Las preguntas son sintéticas, no de usuarios reales.
- **Mejoras:** clasificador de intenciones con ML (TF-IDF + Naive Bayes/SVM) entrenado con este pool;
  tolerancia a errores; memoria entre temas; validación de documentos por imagen.

Referencias técnicas: [NLTK Chat](https://www.nltk.org/api/nltk.chat.util.html) ·
[Python re](https://docs.python.org/3/library/re.html) · [Google Colab](https://colab.research.google.com/)